# 05 分層分析與干擾因子

Ch03 發現淋浴使用者的 RR 偏高，但資深疫調人員懷疑是**干擾作用（confounding）**造成的。

> 🌧️ **穿雨衣的人比較容易感冒——所以穿雨衣會害你感冒？** 當然不是！是因為「下雨天」同時讓你穿雨衣、也讓你容易感冒。「下雨天」就是干擾因子。

在我們的案例中，「功能狀態」就是那個「下雨天」——它同時影響住民會不會用淋浴、也影響感染風險。

這堂課：**驗證干擾條件 → 分層 RR → 森林圖 → Mantel-Haenszel 調整 → 同質性檢定**。

In [ ]:
# Google Colab setup -- 若在本機執行可跳過此 cell
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
# --- Step 1: 資料準備 ---
import pathlib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from scipy.stats import chi2_contingency
from epi_learning.metrics import risk_ratio

# -- CJK font setup (避免中文標籤顯示為方框) --
# 掃描系統字型目錄，顯式註冊 CJK 字型（比依賴快取更可靠）
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150

df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)
print(f"全體：{len(df)} 人，感染：{df['infected'].sum()} 人")

In [ ]:
# --- Step 2: 粗 RR 回顧 ---
ct = pd.crosstab(df["shower_use"], df["infected"])
a = int(ct.loc[1, 1])
b = int(ct.loc[1, 0])
c = int(ct.loc[0, 1])
d = int(ct.loc[0, 0])

crude_rr = risk_ratio(a, a + b, c, c + d)
print(f"粗 RR (shower_use → infected) = {crude_rr:.3f}")
print(f"  暴露組風險: {a/(a+b):.1%}  未暴露組風險: {c/(c+d):.1%}")

## DAG（有向無環圖）——因果關係地圖

懷疑 `functional_status`（功能狀態）是干擾因子。DAG 用箭頭表示「誰影響誰」：

```
functional_status → shower_use → infected
functional_status ─────────────→ infected
```

- **直接路徑**（我們想研究的）：淋浴使用 → 感染
- **後門路徑**（干擾路徑）：淋浴 ← 功能狀態 → 感染

> 後門路徑就像考試時隔壁同學偷看你的答案——他的分數看起來跟你有關，但其實是因為「坐你旁邊」這個共同原因。分層分析就是把「坐旁邊的」和「沒坐旁邊的」分開看。

干擾因子三要件（少一個都不算）：
1. C 與**暴露**有關聯（能走動的人才會去淋浴）
2. C 與**結果**有關聯（能走動的人活動範圍大、暴露機會多）
3. C **不是**暴露→結果路徑上的中間步驟

### 如何發現潛在的干擾因子？

故事開頭是「資深疫調人員」憑經驗提出功能狀態可能是干擾因子。但不可能每次都靠資深人員——有沒有更系統性的方法？

| 方法 | 做法 | 優點 | 限制 |
|------|------|------|------|
| **文獻回顧** | 查過去類似疫情調查的論文，看別人控制了哪些干擾因子 | 站在前人肩膀上 | 新型疾病可能沒有先例 |
| **畫 DAG** | 根據領域知識畫因果關係圖，找出「後門路徑」 | 邏輯清晰，能區分干擾因子 vs. 中間變項 | 需要對因果機制有基本理解 |
| **統計篩選** | 檢查候選變項是否同時與暴露和結果顯著相關（三要件 #1 + #2） | 用數據佐證，不完全靠直覺 | 統計顯著 ≠ 因果 |
| **Change-in-estimate** | 在模型中加入/移除候選變項，看 RR 或 OR 是否改變 ≥ 10% | 直接回答「它有沒有在干擾」 | 需要先有迴歸模型（→ Ch06） |
| **專家諮詢** | 請教臨床醫師、感管人員、資深流行病學家 | 能捕捉統計抓不到的實務因素 | 主觀，可能有遺漏 |

> 💡 **實務推薦「文獻 + DAG + 統計篩選」三管齊下**。先查文獻列出候選名單 → 畫 DAG 標示因果方向 → 用數據驗證干擾三要件。最後再請資深人員 review。

In [ ]:
# --- Step 3: 驗證干擾三要件 ---
# 在分層之前，先確認「功能狀態」是不是真的符合干擾因子三條件

# 要件 1：functional_status 與 shower_use 有關聯？
print("=== 要件 1：功能狀態 × 淋浴使用率 ===")
print(pd.crosstab(df["functional_status"], df["shower_use"],
                  margins=True, normalize="index").round(3))

print()

# 要件 2：functional_status 與 infected 有關聯？
print("=== 要件 2：功能狀態 × 感染率 ===")
print(pd.crosstab(df["functional_status"], df["infected"],
                  margins=True, normalize="index").round(3))

# 要件 3（邏輯判斷）：功能狀態不是淋浴→感染路徑的中間步驟
# 一個人不會因為「先用了淋浴」才變成能走動的——因果方向不對
# → 三個條件都滿足，可以進行分層分析！
print("\n要件 3：功能狀態不在 淋浴→感染 的因果路徑上 ✓")
print("→ 三要件都滿足，確認是干擾因子")

In [ ]:
# --- Step 4: 分層分析 ---
# 按 functional_status 分層，各層分別計算 shower_use 的 RR

strata = sorted(df["functional_status"].unique())
stratum_results = []

for s in strata:
    sub = df[df["functional_status"] == s]
    ct_s = pd.crosstab(sub["shower_use"], sub["infected"])

    if ct_s.shape != (2, 2):
        print(f"  {s}: 跳過（缺少某些組合）")
        continue

    a_s = int(ct_s.loc[1, 1])
    b_s = int(ct_s.loc[1, 0])
    c_s = int(ct_s.loc[0, 1])
    d_s = int(ct_s.loc[0, 0])
    n_s = a_s + b_s + c_s + d_s

    rr_s = risk_ratio(a_s, a_s + b_s, c_s, c_s + d_s)

    # 95% CI
    ln_rr = np.log(rr_s)
    se = np.sqrt(1/a_s - 1/(a_s+b_s) + 1/c_s - 1/(c_s+d_s))
    ci_lo = np.exp(ln_rr - 1.96 * se)
    ci_hi = np.exp(ln_rr + 1.96 * se)

    stratum_results.append({
        "stratum": s, "n": n_s,
        "a": a_s, "b": b_s, "c": c_s, "d": d_s,
        "RR": rr_s, "CI_lower": ci_lo, "CI_upper": ci_hi,
    })

results_df = pd.DataFrame(stratum_results)
print("=== 分層 RR ===")
for _, row in results_df.iterrows():
    print(f"  {row['stratum']:20s}  RR={row['RR']:.3f}  "
          f"(95% CI: {row['CI_lower']:.3f}–{row['CI_upper']:.3f})  n={row['n']}")
print(f"\n  粗 RR = {crude_rr:.3f}")

In [ ]:
# --- Step 5: 森林圖 ---
fig, ax = plt.subplots(figsize=(8, 4))
y_pos = range(len(results_df))

ax.errorbar(
    results_df["RR"], y_pos,
    xerr=[results_df["RR"] - results_df["CI_lower"],
          results_df["CI_upper"] - results_df["RR"]],
    fmt="o", color="#2c7fb8", capsize=4, markersize=8,
)
ax.axvline(x=1, color="gray", linestyle="--", alpha=0.5)
ax.axvline(x=crude_rr, color="red", linestyle=":", alpha=0.7,
           label=f"粗 RR={crude_rr:.2f}")
ax.set_yticks(list(y_pos))
ax.set_yticklabels(results_df["stratum"])
ax.set_xlabel("Risk Ratio (RR)")
ax.set_title("分層分析森林圖：淋浴使用 → 感染（按功能狀態分層）")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- Step 6: Mantel-Haenszel 加權 RR ---
# 原理：人數多的層權重大，人數少的層權重小
# 就像學期成績不能把小考和期末考直接平均——期末考佔比要大一點
numerator = 0
denominator = 0

for _, row in results_df.iterrows():
    a_i, b_i, c_i, d_i = row["a"], row["b"], row["c"], row["d"]
    n_i = a_i + b_i + c_i + d_i
    numerator += a_i * (c_i + d_i) / n_i
    denominator += c_i * (a_i + b_i) / n_i

rr_mh = numerator / denominator

print(f"Mantel-Haenszel 調整後 RR = {rr_mh:.3f}")
print(f"粗 RR                     = {crude_rr:.3f}")
print(f"差異                      = {crude_rr - rr_mh:.3f}")

# --- 用 10% 法則判斷是否有干擾作用 ---
# |粗 RR - 調整 RR| / 調整 RR >= 10% → 有干擾
change_pct = abs(crude_rr - rr_mh) / rr_mh * 100
print(f"\n變化幅度 = {change_pct:.1f}%（10% 法則門檻）")
if change_pct >= 10:
    print("→ 變化 ≥ 10%，粗 RR 被干擾作用影響了！")
    if crude_rr > rr_mh:
        print("  干擾方向：膨脹（粗 RR 偏高，像漢堡裡塞了太多生菜）")
    else:
        print("  干擾方向：壓抑（粗 RR 偏低，像冰塊壓住溫度計）")
else:
    print("→ 變化 < 10%，干擾作用不明顯")

In [ ]:
# --- Step 7: 同質性檢定——有沒有交互作用？ ---
# 交互作用（effect modification）：暴露的影響「因人而異」
# 比喻：調查「吃麻辣鍋會不會拉肚子」
#   胃好的人 RR=1.2，胃不好的人 RR=4.5 → 這不是干擾，是交互作用
#   不能只報合併 RR，要分開說

rr_values = results_df["RR"].values
rr_range = rr_values.max() - rr_values.min()

print("=== 同質性評估 ===")
for _, row in results_df.iterrows():
    print(f"  {row['stratum']:20s}  RR = {row['RR']:.3f}  "
          f"(95% CI: {row['CI_lower']:.3f}–{row['CI_upper']:.3f})")
print(f"\n  RR 範圍：{rr_values.min():.3f} – {rr_values.max():.3f}")
print(f"  變異幅度：{rr_range:.3f}")

if rr_range > 0.5:
    print("\n→ 各層 RR 差異較大，可能存在效果修飾（effect modification）")
    print("  像三家分店辣度差超多，不能只給平均辣度 → 分層報告每組 RR")
else:
    print("\n→ 各層 RR 相近，可合理使用 MH 加權合併值")
    print("  像三家分店咖啡味道一致 → 報一個品牌平均評分即可")

In [ ]:
# --- Step 8: 第二個範例 — 按樓層分層 ---
print("=== 按樓層分層分析：淋浴 → 感染 ===")

for floor in sorted(df["floor"].unique()):
    sub = df[df["floor"] == floor]
    ct_f = pd.crosstab(sub["shower_use"], sub["infected"])
    if ct_f.shape != (2, 2):
        print(f"  {floor}F: 跳過")
        continue
    a_f, b_f = int(ct_f.loc[1, 1]), int(ct_f.loc[1, 0])
    c_f, d_f = int(ct_f.loc[0, 1]), int(ct_f.loc[0, 0])
    rr_f = risk_ratio(a_f, a_f + b_f, c_f, c_f + d_f)
    print(f"  {floor}F: RR={rr_f:.3f}  "
          f"(shower: {a_f}/{a_f+b_f}, no shower: {c_f}/{c_f+d_f})")

print(f"\n  粗 RR = {crude_rr:.3f}")

## 補充：病例對照研究也能用分層分析嗎？

我們的護理之家資料是**世代研究**——追蹤全部 280 位住民，用侵襲率算 RR。但如果是**病例對照研究**（挑病例 + 對照，回頭問暴露史），算不出侵襲率，**只能算 OR（勝算比）**。

好消息：**分層分析的邏輯完全一樣**——驗三要件、按干擾因子分層、用 MH 合併。唯一差異：

| | 世代研究（本章） | 病例對照研究 |
|---|---|---|
| **效應測量** | RR（風險比） | OR（勝算比） |
| **各層計算** | RR = [a/(a+b)] / [c/(c+d)] | OR = (a·d) / (b·c) |
| **MH 合併** | RR_MH = Σ[a·(c+d)/N] / Σ[c·(a+b)/N] | OR_MH = Σ(a·d/N) / Σ(b·c/N) |
| **判斷干擾** | 粗 RR vs. 調整 RR（10% 法則） | 粗 OR vs. 調整 OR（10% 法則） |

> 💡 **口訣**：世代 → MH adjusted **RR**；病例對照 → MH adjusted **OR**。方法一樣，只是換了效應測量。侵襲率低（< 10%）時 OR ≈ RR；侵襲率高（如本案 43%）時 OR 會高估——Ch06 會深入討論。

## 小結

| 步驟 | 學到的技能 | 白話文 |
|------|------------|--------|
| 干擾三要件 | 驗證 C-暴露、C-結果的關聯 | 確認「雙面間諜」的身份 |
| 分層 RR | `crosstab` + 迴圈計算各層 RR | 把大火小火分開比，鎖住火候 |
| 森林圖 | `errorbar` 視覺化各層效應 | 一眼看出各層的 RR 和精確度 |
| MH 調整 | 手算 Mantel-Haenszel RR | 按人數加權的「公平合併」 |
| 10% 法則 | \|粗 RR − 調整 RR\| / 調整 RR | 判斷干擾有沒有大到影響結論 |
| 同質性 | 各層 RR 比較 → 交互作用判斷 | 三家分店味道一不一樣？ |

**限制**：分層分析一次只能控制一個干擾因子。如果同時有年齡、功能狀態、共病等多個干擾因子呢？
→ Ch06 的 **Modified Poisson regression** 可以一次調整所有變項，直接算出 **adjusted RR**。同時也會用邏輯斯迴歸做對照，讓你看到 OR 在高侵襲率下高估了多少。